# 🍽️ Zomato Food Delivery Analysis

#### Python • Exploratory Data Analysis • Power BI Dashboard

In [1]:
import pandas as pd
import numpy as np

df = pd.read_excel('order_history_kaggle_data.xlsx')

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')

Shape: (21321, 29)
Columns: ['Restaurant ID', 'Restaurant name', 'Subzone', 'City', 'Order ID', 'Order Placed At', 'Order Status', 'Delivery', 'Distance', 'Items in order', 'Instructions', 'Discount construct', 'Bill subtotal', 'Packaging charges', 'Restaurant discount (Promo)', 'Restaurant discount (Flat offs, Freebies & others)', 'Gold discount', 'Brand pack discount', 'Total', 'Rating', 'Review', 'Cancellation / Rejection reason', 'Restaurant compensation (Cancellation)', 'Restaurant penalty (Rejection)', 'KPT duration (minutes)', 'Rider wait time (minutes)', 'Order Ready Marked', 'Customer complaint tag', 'Customer ID']


In [2]:
print('=== NULL COUNTS ===')
print(df.isnull().sum())

print('\n=== DTYPES ===')
print(df.dtypes)

print('\n=== ORDER STATUS DISTRIBUTION ===')
print(df['Order Status'].value_counts())

=== NULL COUNTS ===
Restaurant ID                                             0
Restaurant name                                           0
Subzone                                                   0
City                                                      0
Order ID                                                  0
Order Placed At                                           0
Order Status                                              0
Delivery                                                  0
Distance                                                  0
Items in order                                            0
Instructions                                          20601
Discount construct                                     5498
Bill subtotal                                             0
Packaging charges                                         0
Restaurant discount (Promo)                               0
Restaurant discount (Flat offs, Freebies & others)        0
Gold discount       

The following columns are dropped before any further processing:

- `City` and `Delivery` — zero variance (every row is 'Delhi NCR' and 'Zomato Delivery')
- `Instructions` — free-text customer notes, unstructured and not used in any BQ
- `Review` — free-text, heavily null
- `Cancellation / Rejection reason`, `Restaurant compensation`, `Restaurant penalty` — only relevant to non-delivered orders excluded from BQ analysis

`Items in order` is **retained** — fully populated across all 21,321 rows and used in BQ2 basket size analysis.

In [3]:
cols_to_drop = [
    'City', 'Delivery', 'Instructions', 'Review',
    'Cancellation / Rejection reason',
    'Restaurant compensation (Cancellation)',
    'Restaurant penalty (Rejection)',
]
df.drop(columns=cols_to_drop, inplace=True)

print(f'Shape after drop: {df.shape}')

Shape after drop: (21321, 22)


`'Timed out'` (1 row) and `'Picked up'` (3 rows) and `'Return cancelled'` (3 rows) represent ambiguous, incomplete order states.
They have no ratings, no complaint tags, and account for only 0.019% of all orders.
Including them would add noise without contributing to any of the three business questions.

In [4]:
exclude_statuses = ['Timed out', 'Picked up','Return cancelled']

df = df[~df['Order Status'].isin(exclude_statuses)].reset_index(drop=True)

print(f'Shape after exclusion: {df.shape}')
print('Remaining Order Status values:')
print(df['Order Status'].value_counts())

Shape after exclusion: (21314, 22)
Remaining Order Status values:
Order Status
Delivered    21131
Rejected       158
Returned        25
Name: count, dtype: int64


Features (repeat customer counts, total discount sums, complaint rates) will be inflated if any Order IDs are duplicated

In [5]:
print('Duplicate Order IDs:', df.duplicated(subset='Order ID').sum())
df = df.drop_duplicates(subset='Order ID').reset_index(drop=True)

Duplicate Order IDs: 0


Format: `'11:38 PM, September 10 2024'`

`Order Placed At` is parsed to extract two temporal features:

- `Hour` — 0–23, used in BQ1 hourly breakdown to identify peak-hour KPT and rider wait patterns
- `Month` — month name, used in BQ3 to analyse seasonal patterns in discount spend and complaint rates across the 5-month window Sep 2024–Jan 2025

In [6]:
df['Order_Datetime'] = pd.to_datetime(df['Order Placed At'], format='%I:%M %p, %B %d %Y')

df['Hour']        = df['Order_Datetime'].dt.hour
df['Month']       = df['Order_Datetime'].dt.month_name()

# Drop the original raw column and the intermediate datetime (keep only derived features)
df.drop(columns=['Order Placed At', 'Order_Datetime'], inplace=True)

Distance is stored as a string (e.g., `'3km'`, `'<1km'`). 
We convert to a float column `Distance_km`. `'<1km'` is assigned `0.5` as a reasonable midpoint proxy.

In [7]:
print('Unique raw Distance values:', sorted(df['Distance'].unique()))

def parse_distance(val):
    if val == '<1km':
        return 0.5
    return float(val.replace('km', ''))

df['Distance_km'] = df['Distance'].apply(parse_distance).round(1)
df.drop(columns=['Distance'], inplace=True)

print('\nDistance_km summary:')
print(df['Distance_km'].describe().round(2))

Unique raw Distance values: ['10km', '11km', '12km', '13km', '14km', '15km', '16km', '17km', '18km', '19km', '1km', '20km', '21km', '2km', '3km', '4km', '5km', '6km', '7km', '8km', '9km', '<1km']

Distance_km summary:
count    21314.00
mean         4.17
std          2.99
min          0.50
25%          2.00
50%          3.00
75%          6.00
max         21.00
Name: Distance_km, dtype: float64


Extract a clean `Discount_Type` categorical from the raw `Discount construct` column.

1,945 orders have a null `Discount construct` but carry non-zero values in the individual discount columns (primarily `Rest_Discount_Flat` for Swaad orders). These are reclassified as `'Unspecified'` rather than `'No Discount'` to avoid distorting BQ3 analysis.

Five categories: `Percentage`, `Flat Off`, `BOGO`, `No Discount`, `Unspecified`.

In [8]:
def categorize_discount(val):
    if pd.isna(val) or val.strip() == '₹ 0.00':
        return 'No Discount'
    if '%' in val:
        return 'Percentage'
    if 'Flat' in val:
        return 'Flat Off'
    if 'Buy' in val:
        return 'BOGO'
    return 'Other'

df['Discount_Type'] = df['Discount construct'].apply(categorize_discount)
df['Discount construct'] = df['Discount construct'].fillna('No Discount')

print('Discount_Type distribution (before reclassification):')
print(df['Discount_Type'].value_counts())

Discount_Type distribution (before reclassification):
Discount_Type
Percentage     6901
No Discount    5494
BOGO           4620
Flat Off       4297
Other             2
Name: count, dtype: int64


Standardizing column names for usability throughout the project.

In [9]:
df.rename(columns={
    'Restaurant discount (Promo)'                          : 'Rest_Discount_Promo',
    'Restaurant discount (Flat offs, Freebies & others)'   : 'Rest_Discount_Flat',
    'KPT duration (minutes)'                               : 'KPT_mins',
    'Rider wait time (minutes)'                            : 'Rider_Wait_mins',
    'Customer complaint tag'                               : 'Complaint_Tag',
    'Bill subtotal'                                        : 'Bill_Subtotal',
    'Packaging charges'                                    : 'Packaging_Charges',
    'Brand pack discount'                                  : 'Brand_Pack_Discount',
    'Gold discount'                                        : 'Gold_Discount',
    'Discount construct'                                   : 'Discount_Construct',
    'Order Ready Marked'                                   : 'Order_Ready_Marked',
    'Order Status'                                         : 'Order_Status',
    'Restaurant name'                                      : 'Restaurant_Name',
    'Restaurant ID'                                        : 'Restaurant_ID',
    'Order ID'                                             : 'Order_ID',
    'Customer ID'                                          : 'Customer_ID'
}, inplace=True)

print(list(df.columns))

['Restaurant_ID', 'Restaurant_Name', 'Subzone', 'Order_ID', 'Order_Status', 'Items in order', 'Discount_Construct', 'Bill_Subtotal', 'Packaging_Charges', 'Rest_Discount_Promo', 'Rest_Discount_Flat', 'Gold_Discount', 'Brand_Pack_Discount', 'Total', 'Rating', 'KPT_mins', 'Rider_Wait_mins', 'Order_Ready_Marked', 'Complaint_Tag', 'Customer_ID', 'Hour', 'Month', 'Distance_km', 'Discount_Type']


Sum of all four discount sources. Used in Q2 and Q3 to analyse how discount magnitude affects repeat visits and customer satisfaction.

In [10]:
df['Total_Discount'] = (df['Rest_Discount_Promo'] +
                        df['Rest_Discount_Flat']  +
                        df['Gold_Discount']        +
                        df['Brand_Pack_Discount'])

print('Total_Discount summary:')
print(df['Total_Discount'].describe().round(2))
print(f'\n% orders with any discount: {(df["Total_Discount"] > 0).mean() * 100:.1f}%')

Total_Discount summary:
count    21314.00
mean       100.05
std        145.90
min          0.00
25%          0.00
50%         90.00
75%        120.00
max       7787.00
Name: Total_Discount, dtype: float64

% orders with any discount: 61.1%


1,945 orders have a null `Discount_Construct` but carry non-zero values in the individual discount columns — primarily `Rest_Discount_Flat` for Swaad orders. These are reclassified as `'Unspecified'` after `Total_Discount` is computed.

In [11]:
# Reclassify null Discount_Construct rows that carry actual discount values
# These orders received discounts but the construct description was not recorded
has_discount  = df['Total_Discount'] > 0
null_construct = df['Discount_Type'] == 'No Discount'

df.loc[has_discount & null_construct, 'Discount_Type']    = 'Unspecified'
df.loc[has_discount & null_construct, 'Discount_Construct'] = 'Unspecified'

print('Discount_Type distribution (after reclassification):')
print(df['Discount_Type'].value_counts())
print(f"Unspecified avg discount: {df[df['Discount_Type']=='Unspecified']['Total_Discount'].mean():.2f}")

Discount_Type distribution (after reclassification):
Discount_Type
Percentage     6901
BOGO           4620
Flat Off       4297
No Discount    3549
Unspecified    1945
Other             2
Name: count, dtype: int64
Unspecified avg discount: 341.55


The `Total` column holds the final billed amount after all discounts and packaging charges are applied. It is carried forward as a reference variable for Q3 spend analysis. We verify it contains no nulls and no negative values before proceeding.


In [12]:
print('=== Total column validation ===')
print(f'Nulls         : {df["Total"].isnull().sum()}')
print(f'Negative rows : {(df["Total"] < 0).sum()}')
print()
print(df['Total'].describe().round(2))


=== Total column validation ===
Nulls         : 0
Negative rows : 0

count    21314.00
mean       682.59
std        465.31
min         52.50
25%        387.52
50%        597.45
75%        837.90
max      12663.00
Name: Total, dtype: float64


`Complaint_Tag` contains 5 distinct complaint types with NaN for orders where no complaint was logged.

In [13]:
df['Complaint_Category'] = df['Complaint_Tag'].fillna('No Complaint')

print('Complaint_Category distribution:')
print(df['Complaint_Category'].value_counts())

Complaint_Category distribution:
Complaint_Category
No Complaint                        20845
Non-refunded complaint                157
Poor taste or quality                 120
Poor packaging or spillage            104
Wrong item(s) delivered                48
Item(s) missing or not delivered       40
Name: count, dtype: int64


`Customer_Order_Count` — total number of orders placed by that customer in the dataset
`Is_Repeat_Customer` — boolean flag, True if the customer has placed 2 or more orders

In [14]:
df['Customer_Order_Count']= df.groupby('Customer_ID')['Order_ID'].transform('count')

df['Is_Repeat_Customer']   = df['Customer_Order_Count'] >= 2

print('Repeat customer breakdown:')
print(df.drop_duplicates('Customer_ID')['Is_Repeat_Customer'].value_counts())
print(f'\nRepeat customer rate: {df["Is_Repeat_Customer"].mean()*100:.1f}% of all orders')
print(f'\nCustomer_Order_Count distribution:')
print(df.drop_duplicates('Customer_ID')['Customer_Order_Count'].describe().round(2))

Repeat customer breakdown:
Is_Repeat_Customer
False    7716
True     3891
Name: count, dtype: int64

Repeat customer rate: 63.8% of all orders

Customer_Order_Count distribution:
count    11607.00
mean         1.84
std          2.03
min          1.00
25%          1.00
50%          1.00
75%          2.00
max         61.00
Name: Customer_Order_Count, dtype: float64


`Order_Ready_Marked` contains three values: `'Correctly'`, `'Incorrectly'`, and `'Missed'`.

`Order_Ready_Compliant` — a binary boolean where `True` = `'Correctly'` and `False` = `'Incorrectly'` or `'Missed'`.

In [15]:
print('Order_Ready_Marked distribution (including nulls):')
print(df['Order_Ready_Marked'].value_counts(dropna=False))

df['Order_Ready_Compliant'] = df['Order_Ready_Marked'].map({
    'Correctly'   : True,
    'Incorrectly' : False,
    'Missed'      : False
})

print('\nOrder_Ready_Compliant distribution:')
print(df['Order_Ready_Compliant'].value_counts(dropna=False))

Order_Ready_Marked distribution (including nulls):
Order_Ready_Marked
Correctly      19082
Incorrectly     1895
Missed           337
Name: count, dtype: int64

Order_Ready_Compliant distribution:
Order_Ready_Compliant
True     19082
False     2232
Name: count, dtype: int64


Verify completeness and order volume distribution across all 8 subzones. Used as a dimension in BQ2 basket size analysis and BQ3 discount strategy.

In [16]:
print(f'Subzone nulls       : {df["Subzone"].isnull().sum()}')
print(f'Unique subzones     : {df["Subzone"].nunique()}')

print('\nSubzone distribution:')
print(df['Subzone'].value_counts())

Subzone nulls       : 0
Unique subzones     : 8

Subzone distribution:
Subzone
Greater Kailash 2 (GK2)    7379
Sector 4                   6525
DLF Phase 1                3686
Sector 135                 2441
Vasant Kunj                 920
Shahdara                    360
Chittaranjan Park             2
Sikandarpur                   1
Name: count, dtype: int64



All null KPT and Rider Wait rows in Delivered orders belong to restaurants that have sufficient non-null records — restaurant-level imputation is fully feasible with no fallback needed.

In [17]:
delivered_mask = df['Order_Status'] == 'Delivered'

df.loc[delivered_mask, 'KPT_mins'] = (
    df[delivered_mask]
    .groupby('Restaurant_Name')['KPT_mins']
    .transform(lambda x: x.fillna(x.median()))
)

df.loc[delivered_mask, 'Rider_Wait_mins'] = (
    df[delivered_mask]
    .groupby('Restaurant_Name')['Rider_Wait_mins']
    .transform(lambda x: x.fillna(x.median()))
)

print('KPT nulls remaining after imputation:')
print(df[df['KPT_mins'].isna()]['Order_Status'].value_counts())

print('\nRider Wait nulls remaining after imputation:')
print(df[df['Rider_Wait_mins'].isna()]['Order_Status'].value_counts())

KPT nulls remaining after imputation:
Order_Status
Rejected    97
Name: count, dtype: int64

Rider Wait nulls remaining after imputation:
Order_Status
Rejected    118
Name: count, dtype: int64


We inspect the distribution of both columns on Delivered orders and cap extreme outliers at the 99th percentile. Values beyond this threshold are likely data entry anomalies and would skew restaurant-level averages in the Power BI dashboard.

In [18]:
print('=== KPT_mins (Delivered orders) ===')
print(df.loc[delivered_mask, 'KPT_mins'].describe().round(2))

kpt_99 = df.loc[delivered_mask, 'KPT_mins'].quantile(0.99)
print(f'\nKPT 99th percentile : {kpt_99:.1f} mins')

print('\n=== Rider_Wait_mins (Delivered orders) ===')
print(df.loc[delivered_mask, 'Rider_Wait_mins'].describe().round(2))

wait_99 = df.loc[delivered_mask, 'Rider_Wait_mins'].quantile(0.99)
print(f'\nRider Wait 99th percentile : {wait_99:.1f} mins')

df.loc[delivered_mask, 'KPT_mins'] = df.loc[delivered_mask, 'KPT_mins'].clip(upper=kpt_99)
df.loc[delivered_mask, 'Rider_Wait_mins'] = df.loc[delivered_mask, 'Rider_Wait_mins'].clip(upper=wait_99)

print('\nPost-cap KPT_mins max      :', round(df.loc[delivered_mask, 'KPT_mins'].max(),2))
print('Post-cap Rider_Wait_mins max:', round(df.loc[delivered_mask, 'Rider_Wait_mins'].max(),2))

=== KPT_mins (Delivered orders) ===
count    21131.00
mean        17.33
std          6.25
min          0.00
25%         13.42
50%         16.33
75%         20.02
max         90.87
Name: KPT_mins, dtype: float64

KPT 99th percentile : 38.7 mins

=== Rider_Wait_mins (Delivered orders) ===
count    21131.00
mean         4.82
std          4.98
min          0.10
25%          1.00
50%          3.10
75%          7.40
max         73.80
Name: Rider_Wait_mins, dtype: float64

Rider Wait 99th percentile : 21.1 mins

Post-cap KPT_mins max      : 38.67
Post-cap Rider_Wait_mins max: 21.07


Rating is null for **88.3%** of orders — this is a structural feature of the data, not a data quality issue. Customers rarely leave ratings.

We do **not** impute ratings. Analysis involving ratings (Q3) will be scoped to the 2,491 rated orders with an explicit note acknowledging the limitation.

In [19]:
df['Rating'] = df['Rating'].astype('Int64')

rated = df['Rating'].notna()

print(f'Total rated orders   : {rated.sum()}')
print(f'Total unrated orders : {(~rated).sum()}')
print(f'Rating coverage      : {rated.mean()*100:.1f}%')

print('\nRating distribution (rated orders only):')
print(df['Rating'].value_counts().sort_index())

Total rated orders   : 2491
Total unrated orders : 18823
Rating coverage      : 11.7%

Rating distribution (rated orders only):
Rating
1     177
2      82
3     144
4     360
5    1728
Name: count, dtype: Int64


Q1: How can rider wait time be reduced by identifying restaurants with consistently long preparation times and poor order-ready marking compliance?

In [20]:
bq1_df = df[df['Order_Status'] == 'Delivered'].copy()

restaurant_summary = (
    bq1_df.groupby('Restaurant_Name')
    .agg(
        Avg_KPT_mins        = ('KPT_mins',             'mean'),
        Avg_Rider_Wait_mins = ('Rider_Wait_mins',       'mean'),
        Compliance_Rate_pct = ('Order_Ready_Compliant', lambda x: x.mean() * 100),
        Total_Orders        = ('Order_ID',              'count')
    )
    .round(2)
    .reset_index()
    .query('Total_Orders >= 50')
    .sort_values('Avg_KPT_mins', ascending=False)
)

print(restaurant_summary.to_string(index=False))

  Restaurant_Name  Avg_KPT_mins  Avg_Rider_Wait_mins  Compliance_Rate_pct  Total_Orders
Tandoori Junction         20.76                 5.89                86.09           151
Dilli Burger Adda         19.20                 4.97                87.44           223
            Swaad         17.74                 4.48                89.76          6282
      Aura Pizzas         16.97                 4.87                90.26         14417


In [21]:
hourly_summary = (
    bq1_df.groupby(['Restaurant_Name', 'Hour'])
    .agg(
        Avg_KPT_mins        = ('KPT_mins',       'mean'),
        Avg_Rider_Wait_mins = ('Rider_Wait_mins', 'mean'),
        Total_Orders        = ('Order_ID',        'count')
    )
    .round(2)
    .reset_index()
    .query('Total_Orders >= 10')
    .sort_values(['Restaurant_Name', 'Avg_KPT_mins'], ascending=[True, False])
)

print(hourly_summary.to_string(index=False))

  Restaurant_Name  Hour  Avg_KPT_mins  Avg_Rider_Wait_mins  Total_Orders
      Aura Pizzas    11         18.67                 7.63           200
      Aura Pizzas    12         18.05                 6.31           617
      Aura Pizzas    20         18.05                 4.10          2044
      Aura Pizzas    13         17.51                 4.97           759
      Aura Pizzas    21         17.50                 4.00          1586
      Aura Pizzas    22         17.10                 4.66          1210
      Aura Pizzas    19         16.95                 4.80          1713
      Aura Pizzas    14         16.88                 5.33           706
      Aura Pizzas    15         16.85                 6.21           532
      Aura Pizzas    16         16.65                 5.82           573
      Aura Pizzas     1         16.60                 5.32           535
      Aura Pizzas     0         16.41                 4.22           623
      Aura Pizzas    17         16.33              

In [22]:
with pd.ExcelWriter('BQ1_Restaurant_Summary.xlsx') as writer:
    restaurant_summary.to_excel(writer, sheet_name='Restaurant_Summary', index=False)
    hourly_summary.to_excel(writer, sheet_name='Hourly_Breakdown', index=False)

print('Exported BQ1_Restaurant_Summary.xlsx')
print('Restaurant_Summary shape:', restaurant_summary.shape)
print('Hourly_Breakdown shape  :', hourly_summary.shape)

Exported BQ1_Restaurant_Summary.xlsx
Restaurant_Summary shape: (4, 5)
Hourly_Breakdown shape  : (51, 5)


## BQ2: Order Value Optimisation — Basket Size Analysis

**BQ2:** *How can Zomato increase average order value by identifying which menu items and item combinations drive higher basket sizes, and what app-level cross-selling recommendations and order value incentives can convert single-item orders into multi-item orders?*

All 21,321 orders are used. `Items in order` is parsed to extract individual item names and quantities per order.

Parse `Items in order` into structured item lists. Derive `Item_Count` (total quantity of items) and `Unique_Items` (number of distinct items) per order.

In [23]:
import re
from collections import Counter

def parse_items(item_str):
    items = []
    for part in item_str.split(','):
        part = part.strip()
        match = re.match(r'(\d+)\s*x\s*(.+)', part)
        if match:
            items.append((match.group(2).strip(), int(match.group(1))))
    return items

df['Parsed_Items']  = df['Items in order'].apply(parse_items)
df['Item_Count']    = df['Parsed_Items'].apply(lambda x: sum(q for _, q in x))
df['Unique_Items']  = df['Parsed_Items'].apply(len)

print('Item_Count distribution:')
print(df['Item_Count'].describe().round(2))
print(f'\nSingle item orders : {(df["Unique_Items"]==1).sum()}')
print(f'Multi item orders  : {(df["Unique_Items"]>1).sum()}')
print(f'Single item avg bill: {df[df["Unique_Items"]==1]["Bill_Subtotal"].mean():.2f}')
print(f'Multi item avg bill : {df[df["Unique_Items"]>1]["Bill_Subtotal"].mean():.2f}')

Item_Count distribution:
count    21314.00
mean         1.94
std          1.18
min          1.00
25%          1.00
50%          2.00
75%          2.00
max         29.00
Name: Item_Count, dtype: float64

Single item orders : 9624
Multi item orders  : 11690
Single item avg bill: 523.25
Multi item avg bill : 936.82


Analyse how item count directly impacts order value and discount received.

In [24]:
item_value = (
    df.groupby('Item_Count')
    .agg(
        Orders       = ('Order_ID',      'count'),
        Avg_Bill     = ('Bill_Subtotal',  'mean'),
        Avg_Discount = ('Total_Discount', 'mean'),
        Avg_Total    = ('Total',          'mean')
    )
    .round(2)
    .query('Orders >= 10')
)

print('Item Count vs Order Value:')
print(item_value.to_string())

Item Count vs Order Value:
            Orders  Avg_Bill  Avg_Discount  Avg_Total
Item_Count                                           
1             8351    509.36         86.38     444.20
2             9239    722.84         88.99     665.60
3             1816   1203.25        157.23    1098.42
4             1349   1225.19        124.21    1156.08
5              214   1824.94        185.98    1721.09
6              209   1736.29        161.71    1653.31
7               34   2541.66        357.36    2293.59
8               47   2522.50        295.73    2338.36
9               12   3600.33        543.76    3209.41
10              18   3527.22        198.41    3495.25


Identify the most common item pairs in multi-item orders — these natural pairings are the basis for cross-selling prompts.

In [25]:
pairs = Counter()
for items in df[df['Unique_Items'] > 1]['Parsed_Items']:
    names = [i[0] for i in items]
    for i in range(len(names)):
        for j in range(i+1, len(names)):
            pair = tuple(sorted([names[i], names[j]]))
            pairs[pair] += 1

print('Top 10 most common item pairs:')
for pair, count in pairs.most_common(10):
    print(f'  {count:4d}  {pair[0]}  +  {pair[1]}')

Top 10 most common item pairs:
   484  Bageecha Pizza  +  Chilli Cheese Garlic Bread
   371  Bageecha Pizza  +  Makhani Paneer Pizza
   223  All About Chicken Pizza  +  Bageecha Pizza
   213  Bageecha Pizza  +  Cheesy Garlic Bread
   212  Bageecha Pizza  +  Herbed Potato
   189  Chilli Cheese Garlic Bread  +  Makhani Paneer Pizza
   161  Bone in Jamaican Grilled Chicken  +  Bone in Peri Peri Grilled Chicken
   159  Bageecha Pizza  +  Margherita Pizza
   158  Bone in Jamaican Grilled Chicken  +  Bone in Smoky Bbq Grilled Chicken
   157  Jamaican Chicken Melt  +  Murgh Amritsari Seekh Melt


Identify which restaurants have the highest proportion of single-item orders — these are the primary targets for cross-selling intervention.

In [26]:
restaurant_basket = (
    df.groupby('Restaurant_Name')
    .agg(
        Total_Orders      = ('Order_ID',      'count'),
        Single_Item_Orders= ('Unique_Items',  lambda x: (x==1).sum()),
        Single_Item_Pct   = ('Unique_Items',  lambda x: (x==1).mean()*100),
        Avg_Bill          = ('Bill_Subtotal',  'mean'),
        Avg_Items         = ('Item_Count',    'mean')
    )
    .round(2)
    .sort_values('Single_Item_Pct', ascending=False)
)

print('Single Item Order Rate by Restaurant:')
print(restaurant_basket.to_string())

Single Item Order Rate by Restaurant:
                      Total_Orders  Single_Item_Orders  Single_Item_Pct  Avg_Bill  Avg_Items
Restaurant_Name                                                                             
Masala Junction                 27                  22            81.48    353.59       1.19
The Chicken Junction            32                  21            65.62    425.62       1.69
Tandoori Junction              154                 101            65.58    818.83       1.51
Swaad                         6332                3115            49.19    644.01       1.87
Dilli Burger Adda              227                 103            45.37    603.92       1.89
Aura Pizzas                  14542                6262            43.06    799.27       1.97


Analyse basket size by hour and discount type to identify when and under which discount structure customers order more items.

In [27]:
hourly_basket = (
    df.groupby('Hour')
    .agg(
        Orders    = ('Order_ID',     'count'),
        Avg_Items = ('Item_Count',   'mean'),
        Avg_Bill  = ('Bill_Subtotal', 'mean')
    )
    .round(2)
)

discount_basket = (
    df.groupby('Discount_Type')
    .agg(
        Orders    = ('Order_ID',     'count'),
        Avg_Items = ('Item_Count',   'mean'),
        Avg_Bill  = ('Bill_Subtotal', 'mean')
    )
    .round(2)
    .sort_values('Avg_Items', ascending=False)
)

print('Basket Size by Hour:')
print(hourly_basket.to_string())
print('\nBasket Size by Discount Type:')
print(discount_basket.to_string())

Basket Size by Hour:
      Orders  Avg_Items  Avg_Bill
Hour                             
0        956       1.72    696.42
1        833       1.74    678.36
2        488       1.73    677.29
3        389       1.65    664.76
4          5       1.40    505.20
11       305       2.22    851.62
12       909       1.96    764.84
13      1142       1.91    772.08
14      1032       1.90    738.11
15       824       1.88    700.74
16       904       1.86    697.66
17      1069       1.91    715.90
18      1611       1.98    755.47
19      2418       2.10    814.88
20      2911       2.05    797.09
21      2295       1.96    757.98
22      1748       1.91    733.65
23      1475       1.85    720.77

Basket Size by Discount Type:
               Orders  Avg_Items  Avg_Bill
Discount_Type                             
Other               2       3.00    817.00
BOGO             4620       2.47    608.46
Flat Off         4297       2.07    965.78
Unspecified      1945       1.92    820.49
No Discoun

Summary insight prints for BQ2.

In [28]:
top_pair = pairs.most_common(1)[0]
best_hour = hourly_basket['Avg_Items'].idxmax()
worst_disc = discount_basket['Avg_Items'].idxmin()
single_pct = (df['Unique_Items']==1).mean()*100

print(f'Single item orders          : {(df["Unique_Items"]==1).sum()} ({single_pct:.1f}% of all orders)')
print(f'Single item avg bill        : {df[df["Unique_Items"]==1]["Bill_Subtotal"].mean():.2f}')
print(f'Multi item avg bill         : {df[df["Unique_Items"]>1]["Bill_Subtotal"].mean():.2f}')
print(f'Top item pair               : {top_pair[0][0]}  +  {top_pair[0][1]} ({top_pair[1]} orders)')
print(f'Highest basket size hour    : {best_hour}:00 ({hourly_basket.loc[best_hour,"Avg_Items"]:.2f} avg items)')
print(f'Lowest basket size disc type: {worst_disc} ({discount_basket.loc[worst_disc,"Avg_Items"]:.2f} avg items)')

Single item orders          : 9624 (45.2% of all orders)
Single item avg bill        : 523.25
Multi item avg bill         : 936.82
Top item pair               : Bageecha Pizza  +  Chilli Cheese Garlic Bread (484 orders)
Highest basket size hour    : 11:00 (2.22 avg items)
Lowest basket size disc type: Percentage (1.61 avg items)


Export item-level and basket summary tables for Power BI dashboard.

In [29]:
# Top item pairs as exportable dataframe
pairs_df = pd.DataFrame(
    [(f'{p[0]}  +  {p[1]}', c) for p, c in pairs.most_common(20)],
    columns=['Item_Pair', 'Co_Order_Count']
)

with pd.ExcelWriter('BQ2_Basket_Analysis.xlsx') as writer:
    item_value.reset_index().to_excel(writer,       sheet_name='Item_Count_vs_Value',   index=False)
    restaurant_basket.reset_index().to_excel(writer, sheet_name='Restaurant_Basket',     index=False)
    pairs_df.to_excel(writer,                         sheet_name='Top_Item_Pairs',        index=False)
    hourly_basket.reset_index().to_excel(writer,     sheet_name='Hourly_Basket',         index=False)
    discount_basket.reset_index().to_excel(writer,   sheet_name='Discount_Basket',       index=False)

print('Exported BQ2_Basket_Analysis.xlsx')
print('Sheets: Item_Count_vs_Value, Restaurant_Basket, Top_Item_Pairs, Hourly_Basket, Discount_Basket')

Exported BQ2_Basket_Analysis.xlsx
Sheets: Item_Count_vs_Value, Restaurant_Basket, Top_Item_Pairs, Hourly_Basket, Discount_Basket


## BQ3: Discount Strategy Optimisation

**BQ3:** *How can discount strategy be optimized by analysing whether high discount spend reduces customer complaints and improves satisfaction across restaurants?*

Three dimensions are analysed: restaurant level, discount type, and monthly trends. `Rating` is used as the satisfaction indicator alongside `Complaint_Category` as the dissatisfaction indicator.

Restaurant-level analysis of discount spend, complaint rate, and average rating — the primary unit of analysis for BQ3.

In [30]:
restaurant_q3 = (
    df.groupby('Restaurant_Name')
    .agg(
        Total_Orders     = ('Order_ID',           'count'),
        Avg_Discount     = ('Total_Discount',      'mean'),
        Pct_Discounted   = ('Total_Discount',      lambda x: (x > 0).mean() * 100),
        Complaint_Rate   = ('Complaint_Category',  lambda x: (x != 'No Complaint').mean() * 100),
        Avg_Rating       = ('Rating',              'mean'),
        Total_Complaints = ('Complaint_Category',  lambda x: (x != 'No Complaint').sum())
    )
    .round(2)
    .query('Total_Orders >= 50')
    .sort_values('Complaint_Rate', ascending=False)
    .reset_index()
)

print(restaurant_q3.to_string(index=False))

  Restaurant_Name  Total_Orders  Avg_Discount  Pct_Discounted  Complaint_Rate  Avg_Rating  Total_Complaints
Dilli Burger Adda           227        177.19           92.07            3.52        4.18                 8
      Aura Pizzas         14542         95.45           58.05            2.44        4.32               355
            Swaad          6332        110.75           68.60            1.66        4.43               105
Tandoori Junction           154          0.00            0.00            0.00        4.65                 0


Discount type vs complaint rate and satisfaction — identifies which discount structure performs best and worst.

In [31]:
discount_complaint = (
    df.groupby('Discount_Type')
    .agg(
        Total_Orders   = ('Order_ID',           'count'),
        Complaint_Rate = ('Complaint_Category',  lambda x: (x != 'No Complaint').mean() * 100),
        Avg_Discount   = ('Total_Discount',      'mean'),
        Avg_Rating     = ('Rating',              'mean')
    )
    .round(2)
    .sort_values('Complaint_Rate', ascending=False)
    .reset_index()
)

print(discount_complaint.to_string(index=False))

Discount_Type  Total_Orders  Complaint_Rate  Avg_Discount  Avg_Rating
         BOGO          4620            2.62          0.00         4.4
     Flat Off          4297            2.40        148.30        4.33
   Percentage          6901            2.38        120.40        4.28
  Unspecified          1945            2.01        341.55        4.38
  No Discount          3549            1.18          0.00        4.47
        Other             2            0.00          0.00        <NA>


Monthly discount spend vs complaint rate across the 5-month window (Sep 2024 to Jan 2025).
Order volume is flat across all months (~4,200 orders each), so complaint rate variation is driven by discount levels and service quality rather than volume effects.

In [32]:
monthly_q3 = (
    df.groupby('Month')
    .agg(
        Total_Orders   = ('Order_ID',           'count'),
        Avg_Discount   = ('Total_Discount',      'mean'),
        Complaint_Rate = ('Complaint_Category',  lambda x: (x != 'No Complaint').mean() * 100),
        Avg_Rating     = ('Rating',             'mean')
    )
    .round(2)
    .reset_index()
)

month_order = ['September', 'October', 'November', 'December', 'January']
monthly_q3['Month'] = pd.Categorical(monthly_q3['Month'], categories=month_order, ordered=True)
monthly_q3 = monthly_q3.sort_values('Month')

print(monthly_q3.to_string(index=False))

    Month  Total_Orders  Avg_Discount  Complaint_Rate  Avg_Rating
September          4238        115.34            1.37        4.32
  October          4276         76.35            2.43        4.34
 November          4491         68.13            2.78        4.23
 December          4300        131.37            2.07        4.44
  January          4009        111.32            2.32        4.42


Summary insight prints for BQ3.

In [33]:
highest_complaint  = restaurant_q3.iloc[0]
lowest_complaint   = restaurant_q3.iloc[-1]
lowest_complaint_type = (discount_complaint
                          [discount_complaint['Discount_Type'] != 'Other']
                          .sort_values('Complaint_Rate').iloc[0])
highest_discount_type = discount_complaint.sort_values('Avg_Discount', ascending=False).iloc[0]

print(f'Highest complaint rate restaurant    : {highest_complaint["Restaurant_Name"]} ({highest_complaint["Complaint_Rate"]}%)')
print(f'Lowest complaint rate restaurant     : {lowest_complaint["Restaurant_Name"]} ({lowest_complaint["Complaint_Rate"]}%)')
print(f'Highest avg discount restaurant      : {restaurant_q3.sort_values("Avg_Discount", ascending=False).iloc[0]["Restaurant_Name"]}')
print(f'Discount type with lowest complaints : {lowest_complaint_type["Discount_Type"]} ({lowest_complaint_type["Complaint_Rate"]}%)')
print(f'Discount type with highest avg disc  : {highest_discount_type["Discount_Type"]} (Rs.{highest_discount_type["Avg_Discount"]})')
print(f'Best month (lowest complaints)       : {monthly_q3.loc[monthly_q3["Complaint_Rate"].idxmin(), "Month"]}')
print(f'Worst month (highest complaints)     : {monthly_q3.loc[monthly_q3["Complaint_Rate"].idxmax(), "Month"]}')

Highest complaint rate restaurant    : Dilli Burger Adda (3.52%)
Lowest complaint rate restaurant     : Tandoori Junction (0.0%)
Highest avg discount restaurant      : Dilli Burger Adda
Discount type with lowest complaints : No Discount (1.18%)
Discount type with highest avg disc  : Unspecified (Rs.341.55)
Best month (lowest complaints)       : September
Worst month (highest complaints)     : November


Export three-sheet summary for Power BI dashboard.

In [34]:
with pd.ExcelWriter('BQ3_Discount_Strategy.xlsx') as writer:
    restaurant_q3.to_excel(writer,      sheet_name='Restaurant_Summary',    index=False)
    discount_complaint.to_excel(writer,  sheet_name='Discount_Type_Analysis', index=False)
    monthly_q3.to_excel(writer,          sheet_name='Monthly_Trends',         index=False)

print('Exported BQ3_Discount_Strategy.xlsx')
print('Restaurant_Summary shape   :', restaurant_q3.shape)
print('Discount_Type_Analysis shape:', discount_complaint.shape)
print('Monthly_Trends shape       :', monthly_q3.shape)

Exported BQ3_Discount_Strategy.xlsx
Restaurant_Summary shape   : (4, 7)
Discount_Type_Analysis shape: (6, 5)
Monthly_Trends shape       : (5, 5)
